# Co-Noise Synthesizer Test Notebook
This notebook verifies the `CoNoise` synthesizer implementation, ensuring it correctly adds violations to the data.

## 1. Setup & Loader Orchestration

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.loading.file_loader import FileLoader
from src.loading.components.data_loader import DataLoader
from src.loading.components.dcs_loader import DCsLoader
from src.loading.components.metadata_loader import MetadataLoader
from src.loading.components.data_encoder import DataEncoder
from src.loading.components.dcs_encoder import DCsEncoder
from src.synthesizing.co_noise import CoNoise

def get_dataset(dataset_name):
    loader = FileLoader(
        name=dataset_name, 
        base_path="../data",
        data_loader=DataLoader(),
        dcs_loader=DCsLoader(),
        metadata_loader=MetadataLoader(),
        data_encoder=DataEncoder(),
        dcs_encoder=DCsEncoder()
    )
    return loader.load()

## 2. Co-Noise Execution
We will run Co-Noise on the `adult` dataset (which initially has 0 violations) and verify that violations are introduced.

In [ ]:
dataset = get_dataset("adult")
print(f"Original violations: {len(dataset.get_violations())}")

synthesizer = CoNoise(num_of_iterations=500)
synthetic_dataset = synthesizer.synthesize(dataset)

print(f"Synthetic dataset name: {synthetic_dataset.name}")
print(f"Synthetic violations: {len(synthetic_dataset.get_violations())}")

assert len(synthetic_dataset.get_violations()) > 0, "Co-Noise should have introduced violations!"
print("Logical test passed: Violations introduced successfully.")

## 3. Visualization
Compare original vs synthetic data distributions for a few columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

col = dataset.data.columns[0]
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(dataset.data[col], kde=True)
plt.title(f"Original {col} Distribution")

plt.subplot(1, 2, 2)
sns.histplot(synthetic_dataset.data[col], kde=True)
plt.title(f"Synthetic {col} Distribution (Co-Noise)")

plt.tight_layout()
plt.show()

## 4. Runtime Test

In [ ]:
import time

iterations = [10, 50, 100, 200]
runtimes = []

for n in iterations:
    start = time.time()
    CoNoise(num_of_iterations=n).synthesize(dataset)
    end = time.time()
    runtimes.append(end - start)
    print(f"Iterations: {n}, Time: {end - start:.4f}s")

plt.figure(figsize=(6, 4))
plt.plot(iterations, runtimes, marker='o')
plt.xlabel("Number of Iterations")
plt.ylabel("Runtime (s)")
plt.title("Co-Noise Runtime Scaling")
plt.grid(True)
plt.show()